In [13]:
from typing import List, Literal,TypedDict
from pydantic import BaseModel,Field
from pypdf import PdfReader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_google_genai  import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings 
from langchain_community.tools.tavily_search import TavilySearchResults

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

In [14]:
load_dotenv()

True

In [15]:
#---------------LOAD DOCS-----------------------------------
#Helper function to replicate PyPDFLoader
def load_pdf_as_documents(file_path: str) -> list[Document]:
    try:
        reader = PdfReader(file_path)
        documents = []

        for i, page in enumerate(reader.pages):
            text = page.extract_text() or ""
            # Replicates LangChain's default metadata schema for loaders
            metadata = {"source": file_path, "page": i}
            documents.append(Document(page_content=text, metadata=metadata))

        return documents
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return []


# Combining docs identically
docs = (
    load_pdf_as_documents("./documents/book1.pdf")
    #+ load_pdf_as_documents("./documents/book2.pdf")
    #+ load_pdf_as_documents("./documents/book3.pdf")   
)

In [16]:
len(docs)

2123

In [17]:
#-----------------------------------CHUNKING--------------------------------
#chunking the documents using Recursive Character text splitter
chunks = RecursiveCharacterTextSplitter(chunk_size =900, chunk_overlap =100).split_documents(docs)

#---------------------------Clean text to avoid UnicodeEncodeError-------------------

for d in chunks:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")

In [18]:
len(chunks)

6580

In [19]:
#-----------------------Indexing------------------------------ 
#(fresh collection each run)
#------------------------------EMBED DOCS-----------------------------------
#Generating Embeddings for given docs using text-embedding-004 from google generative ai embeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model = "gemini-embedding-001",# or 'models/gemini-embedding-exp-03-07'
    task_type="retrieval_document"# Options: retrieval_query, retrieval_document, semantic_similarity, etc.
)


In [ ]:
#as the gemini tokens expired using huggingface models -sentence transformers
#pip install langchain langchain-community langchain-faiss sentence-transformers


In [28]:
'''
from langchain_huggingface import HuggingFaceEmbeddings
import sentence_transformers # type: ignore[import]
# Initialize the Hugging Face Model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},                  # Switch to 'cuda' if using GPU
    encode_kwargs={'normalize_embeddings': True}     # Crucial for accurate FAISS cosine similarity
)
'''

NameError: name 'nn' is not defined

In [ ]:
#----------------------------------VECTOR STORE-----------------------------------
#Storing in vector database using  FAISS db
vector_store = FAISS.from_documents(chunks,embeddings)

In [ ]:
#-------------------------RETRIEVER-----------------------------
#Creating a retriever using vector store
retriever =  vector_store.as_retriever(search_type='similarity',search_kwargs ={"k":4})

In [ ]:
#---------------------------LLM-------------------------------
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2)
#llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash",temperature = 0)

In [ ]:
#-------------------- Graph State ---------------------------------------
class State(TypedDict):
    question: str
    docs: List[Document]
    answer: str

In [ ]:
#------------------------------NODES DEFINITION---------------------------------------
def retrieve(state):
    query = state["question"]
    return {"docs": retriever.invoke(query)}

In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer only from the context. If not in context, say you don't know."),
        ("human", "Question: {question}\n\nContext:\n{context}"),
    ]
)
def generate(state):
    context = "\n\n".join(d.page_content for d in state["docs"])
    out = (prompt | llm).invoke({"question": state["question"], "context": context})
    return {"answer": out.content}

In [ ]:
#---------------------------------BUILDING GRAPH-------------------------------------
graph = StateGraph(State)
graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)

graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)

app = graph.compile()

app

In [ ]:
#-------------------------------------Run-----------------------------------------

result = app.invoke({"question": "WHat is a transformer in deep learning.", "docs": [], "answer": ""})
print(result["answer"])

In [ ]:
print(result['docs'][0].page_content)
print('*'*100)
print(result['docs'][1].page_content)
print('*'*100)
print(result['docs'][2].page_content)
print('*'*100)
print(result['docs'][3].page_content)